# Pivoting Stability Explorer

Chapter 5: Square Matrix and LU Decomposition · *Mathematics for AI and Machine Learning* by Fuheng Wu

Compare elimination with and without partial pivoting on near-singular systems.

Companion code for the [interactive demo](https://math4ai.org/companion/chapter-5/pivoting-stability/) on math4ai.org (sign-in required). Run the cell with Shift+Enter; PyTorch is preinstalled on Colab. Change the values at the top and run it again to explore.


In [1]:
import torch

# The demo's system: A = [[eps, 1], [1, 1]], b = [1, 2]; the solution is close to [1, 1]
eps, delta = 0.02, 0.03


def solve_no_pivot(A, b):
    """Gaussian elimination on a 2x2 system, always using A[0,0] as the pivot."""
    A, b = A.clone(), b.clone()
    m = A[1, 0] / A[0, 0]                  # huge when eps is tiny
    A[1] -= m * A[0]
    b[1] -= m * b[0]
    x2 = b[1] / A[1, 1]
    x1 = (b[0] - A[0, 1] * x2) / A[0, 0]
    return torch.stack([x1, x2]), abs(m).item()


def solve_pivot(A, b):
    """Swap rows so the larger entry in column 0 is the pivot."""
    if abs(A[1, 0]) > abs(A[0, 0]):
        A, b = A[[1, 0]], b[[1, 0]]
    return solve_no_pivot(A, b)


b = torch.tensor([1.0, 2.0])
bp = torch.tensor([1 + delta, 2 - delta])
A = torch.tensor([[eps, 1.0], [1.0, 1.0]])
x, growth = solve_no_pivot(A, b)
xp, _ = solve_no_pivot(A, bp)
print(f"eps={eps}: no-pivot multiplier = {growth:.1f}, x = {x.tolist()}")
print("change in x when b moves by delta:", (xp - x).norm().item())

# ── Shrink eps in float32: without pivoting the answer falls apart ──────────
print("\n  eps      no pivot                pivot                 exact (float64)")
for e in [1e-2, 1e-4, 1e-6, 1e-8]:
    A = torch.tensor([[e, 1.0], [1.0, 1.0]], dtype=torch.float32)
    x_np, _ = solve_no_pivot(A, b)
    x_p, _ = solve_pivot(A, b)
    exact = torch.linalg.solve(A.double(), b.double())
    print(f"{e:7.0e}  {str([round(t, 5) for t in x_np.tolist()]):22} "
          f"{str([round(t, 5) for t in x_p.tolist()]):22} {[round(t, 5) for t in exact.tolist()]}")

eps=0.02: no-pivot multiplier = 50.0, x = [1.0204076766967773, 0.9795918464660645]
change in x when b moves by delta: 0.06872829049825668

  eps      no pivot                pivot                 exact (float64)
  1e-02  [1.0101, 0.9899]       [1.0101, 0.9899]       [1.0101, 0.9899]
  1e-04  [1.00017, 0.9999]      [1.0001, 0.9999]       [1.0001, 0.9999]
  1e-06  [1.01328, 1.0]         [1.0, 1.0]             [1.0, 1.0]
  1e-08  [0.0, 1.0]             [1.0, 1.0]             [1.0, 1.0]
